# 🛒 Caso Semestral - RetailSmart
## Análisis de Clientes para una Cadena de Comercio
### Machine Learning - MLY0100 | DUOC UC

---

| Campo | Detalle |
|---|---|
| **Integrante 1** | _Francisco Jesús Aranguiz Inostroza_ |
| **Integrante 2** | _Vicente Benjamin Alarcón Gallardo_ |
| **Sección** | _003D_OLS_ |
| **Docente** | _Joan Manuel Toro Ortiz_ |
| **Fecha de entrega** | **_POR DEFINIR_** |
| **Evaluación** | Parcial 1 - Entrega Grupal (70%) |

---

### 📋 Descripción del trabajo

> _(Escriban aquí un párrafo breve describiendo qué hicieron en este notebook, qué decisiones tomaron y cuáles fueron los hallazgos más relevantes.)_

---
# FASE 1 — Comprensión del Negocio
### `CRISP-DM: Business Understanding`

En esta fase debemos entender el contexto del negocio antes de tocar cualquier dato.  
El objetivo es responder: **¿qué problema queremos resolver y por qué importa?**

## 1.1 Contexto de RetailSmart

> _(Describan con sus propias palabras el contexto del negocio. ¿Qué hace RetailSmart? ¿En qué regiones opera? ¿Qué canales usa? ¿Cuál es el problema que enfrenta?)_

RetailSmart es una empresa del sector minorista que opera en **5 regiones de Chile** (**Metropolitana, Valparaíso, Biobío, Araucanía y Coquimbo**), operando mediante tiendas fisicas, plataforma online y aplicación móvil. La empresa ha acumulado una importante base de datos históricos sobre sus clientes, incluyendo información demográfica y comportamiento de compra. En el entorno competitivo actual, la empresa busca modernizar su estrategia pasando de una toma de decisiones basada en la intuición a un enfoque Data-Driven (Tomar decisiones basandose en los datos recopilados con anterioridad). El desafío principal es utilizar esta información para optimizar las campañas de marketing, mejorar la retención y proyectar el valor económico de sus consumidores.

## 1.2 Objetivos del Proyecto

> _(Definan al menos 3 objetivos analíticos concretos del proyecto. Deben conectar con las tareas de ML que realizarán.)_

- **Objetivo 1:** Pronosticar el gasto anual del cliente. 
- **Objetivo 2:** Predecir el estado de actividad del cliente.
- **Objetivo 3:** Clasificar los segmentos de consumidores.

## 1.3 Identificación de Targets

Para este proyecto se trabajará con tres targets distintos a lo largo del semestre:

| Target | Variable | Tipo de tarea | Justificación |
|---|---|---|---|
| Gasto anual del cliente | `gasto_anual` | Regresión | _Usamos regresión para predecir un número continuo._ |
| Estado del cliente | `cliente_activo` | Clasificación binaria | _Usamos una clasificación binaria ya que trabajaremos con solo DOS opciones._ |
| Segmento del cliente | `segmento_cliente` | Clasificación multiclase | _Usamos esta clasificación multiclase para categorizar a los clientes._ |

> **Para esta entrega (Parcial 1) el foco es el EDA y preprocesamiento, no el modelado.**

---
# FASE 2 — Comprensión de los Datos
### `CRISP-DM: Data Understanding`

En esta fase exploramos el dataset para entender su estructura, calidad y características.  
El objetivo es detectar patrones, anomalías y oportunidades antes de modelar.

## 2.1 Importación de Librerías

In [1]:
# Librerías de manipulación de datos
import pandas as pd
import numpy as np

# Librerías de visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías de preprocesamiento (se usarán en Fase 3)
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer

# Configuraciones generales
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

print('✅ Librerías importadas correctamente')

✅ Librerías importadas correctamente


## 2.2 Carga del Dataset

**Instrucciones para cargar el archivo en Google Colab:**
1. Descarga el archivo `retail_clientes_MLY0100.csv` desde AVA
2. Ejecuta la celda siguiente — aparecerá un botón para seleccionar el archivo
3. Selecciona el CSV descargado
4. El archivo quedará disponible en el entorno de Colab

In [ ]:
# Carga del archivo desde tu computador hacia Google Colab
from google.colab import files

print('📂 Selecciona el archivo retail_clientes_MLY0100.csv')
uploaded = files.upload()

In [ ]:
# Leer el archivo CSV en un DataFrame
df = pd.read_csv('retail_clientes_MLY0100.csv', encoding='utf-8')

print(f'✅ Dataset cargado correctamente')
print(f'   Filas    : {df.shape[0]:,}')
print(f'   Columnas : {df.shape[1]}')

## 2.3 Exploración Inicial

> Antes de analizar, necesitamos entender qué tenemos: dimensiones, tipos de datos y una vista rápida del contenido.

In [ ]:
# Vista de las primeras filas del dataset
df.head(10)

In [ ]:
# Tipos de datos y valores no nulos por columna
df.info()

In [ ]:
# Estadísticos descriptivos de variables numéricas
df.describe().T

In [ ]:
# Estadísticos de variables categóricas
df.describe(include='object')

## 2.4 Análisis de Valores Faltantes (Missing Values)

> _(Luego de ejecutar el código, interpreten los resultados: ¿cuáles variables tienen missing values? ¿Qué porcentaje representan? ¿Puede tener algún significado de negocio que falten esos valores?)_

In [ ]:
# Conteo y porcentaje de valores faltantes por columna
missing = pd.DataFrame({
    'Valores faltantes': df.isnull().sum(),
    'Porcentaje (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['Valores faltantes'] > 0].sort_values('Porcentaje (%)', ascending=False)
print(missing)

# Visualización
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    missing['Porcentaje (%)'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('Porcentaje de Valores Faltantes por Variable', fontsize=13)
    ax.set_ylabel('% Missing')
    ax.set_xlabel('Variable')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

**Interpretación:**
> _(Escriban aquí su análisis de los missing values encontrados.)_

## 2.5 Análisis de Distribuciones

> _(Analicen la distribución de las variables numéricas. ¿Tienen distribución normal? ¿Están sesgadas? ¿Qué implica eso para el preprocesamiento?)_

In [ ]:
# Distribución de todas las variables numéricas
numericas = df.select_dtypes(include=[np.number]).columns.tolist()
# Excluimos columnas binarias y de ID para el histograma
numericas_plot = [c for c in numericas if c not in ['programa_fidelidad', 'cliente_activo']]

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numericas_plot):
    axes[i].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribución: {col}', fontsize=11)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frecuencia')

# Ocultar ejes sobrantes
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de Variables Numéricas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretación:**
> _(¿Qué variables tienen distribución aproximadamente normal? ¿Cuáles están sesgadas? ¿Esto influye en qué técnica de escalamiento usar?)_

## 2.6 Análisis de Variables Categóricas

In [ ]:
# Frecuencia de categorías en variables categóricas
categoricas = df.select_dtypes(include='object').columns.tolist()
categoricas = [c for c in categoricas if c != 'id_cliente']

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(categoricas):
    counts = df[col].value_counts()
    axes[i].bar(counts.index, counts.values, color='coral', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Frecuencia: {col}', fontsize=11)
    axes[i].set_ylabel('Cantidad')
    axes[i].tick_params(axis='x', rotation=20)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de Variables Categóricas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretación:**
> _(¿Hay categorías dominantes? ¿Hay categorías con muy poca frecuencia? ¿Qué implicancias tiene para el modelo?)_

## 2.7 Detección de Valores Atípicos (Outliers)

> _(Usen boxplots para identificar outliers visualmente. Luego calculen los límites con IQR para cuantificarlos.)_

In [ ]:
# Boxplots de variables numéricas para detectar outliers
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numericas_plot):
    axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[i].set_title(f'Boxplot: {col}', fontsize=11)
    axes[i].set_ylabel(col)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Detección de Outliers - Boxplots', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Cuantificación de outliers usando el método IQR
print('--- Outliers detectados por método IQR ---\n')
for col in numericas_plot:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < limite_inf) | (df[col] > limite_sup)).sum()
    pct = round(n_outliers / len(df) * 100, 2)
    if n_outliers > 0:
        print(f'{col:<25} → {n_outliers:>4} outliers ({pct}%) | Límites: [{limite_inf:,.0f} , {limite_sup:,.0f}]')

**Interpretación:**
> _(¿Qué variables tienen outliers? ¿Son errores de datos o valores reales extremos del negocio? ¿Cómo los tratarán en la siguiente fase?)_

## 2.8 Análisis de Correlación

> _(El mapa de calor muestra qué variables numéricas están relacionadas entre sí. Esto es relevante para el modelado posterior.)_

In [ ]:
# Matriz de correlación entre variables numéricas
corr_matrix = df[numericas].corr()

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5, ax=ax
)
ax.set_title('Matriz de Correlación - Variables Numéricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretación:**
> _(¿Qué variables tienen alta correlación con los targets? ¿Hay variables muy correlacionadas entre sí que podrían generar multicolinealidad?)_

---
# FASE 3 — Preparación de los Datos
### `CRISP-DM: Data Preparation`

En esta fase transformamos los datos crudos en un dataset listo para modelar.  
Cada decisión debe estar **justificada** en las celdas markdown.

In [ ]:
# Trabajamos sobre una copia para preservar el dataset original
df_prep = df.copy()
print(f'Copia creada: {df_prep.shape[0]:,} filas x {df_prep.shape[1]} columnas')

## 3.1 Tratamiento de Missing Values

> **Justificación de la estrategia elegida:**  
> _(Expliquen por qué eligieron cada método para cada variable. Opciones comunes: mediana, media, moda, KNN, eliminación de filas. El criterio debe basarse en la distribución y el significado de negocio de cada variable.)_

In [ ]:
# -------------------------------------------------------
# COMPLETEN ESTE BLOQUE con la estrategia que elijan
# -------------------------------------------------------

# Ejemplo para 'satisfaccion' con mediana:
# mediana_satisfaccion = df_prep['satisfaccion'].median()
# df_prep['satisfaccion'].fillna(mediana_satisfaccion, inplace=True)

# Ejemplo para 'meses_cliente' con KNN:
# from sklearn.impute import KNNImputer
# imputer = KNNImputer(n_neighbors=5)
# df_prep[['meses_cliente']] = imputer.fit_transform(df_prep[['meses_cliente']])

# Verifica que no queden missing values
print('Missing values restantes:')
print(df_prep.isnull().sum()[df_prep.isnull().sum() > 0])

## 3.2 Tratamiento de Outliers

> **Justificación de la estrategia elegida:**  
> _(¿Eliminaron filas, aplicaron winsorización, o los dejaron? Justifiquen cada decisión en función del contexto de negocio.)_

In [ ]:
# -------------------------------------------------------
# COMPLETEN ESTE BLOQUE con la estrategia que elijan
# -------------------------------------------------------

# Ejemplo de winsorización para 'ticket_promedio':
# Q1 = df_prep['ticket_promedio'].quantile(0.25)
# Q3 = df_prep['ticket_promedio'].quantile(0.75)
# IQR = Q3 - Q1
# limite_sup = Q3 + 1.5 * IQR
# df_prep['ticket_promedio'] = df_prep['ticket_promedio'].clip(upper=limite_sup)

print(f'Filas después del tratamiento de outliers: {len(df_prep):,}')

## 3.3 Encoding de Variables Categóricas

> **Justificación:**  
> _(¿Qué variables categóricas codificaron? ¿Usaron LabelEncoder u OneHotEncoder? La elección depende de si la variable es nominal u ordinal.)_

In [ ]:
# -------------------------------------------------------
# COMPLETEN ESTE BLOQUE con la estrategia que elijan
# -------------------------------------------------------

# Ejemplo LabelEncoder para variable ordinal:
# le = LabelEncoder()
# df_prep['genero_enc'] = le.fit_transform(df_prep['genero'])

# Ejemplo OneHotEncoder para variable nominal:
# df_prep = pd.get_dummies(df_prep, columns=['region'], prefix='region')

print('Columnas después del encoding:')
print(df_prep.columns.tolist())

## 3.4 Escalamiento de Variables Numéricas

> **Justificación:**  
> _(¿Usaron StandardScaler o MinMaxScaler? La decisión debe basarse en la distribución de los datos observada en la Fase 2. Variables con distribución normal → StandardScaler. Variables acotadas o con outliers → MinMaxScaler.)_

In [ ]:
# -------------------------------------------------------
# COMPLETEN ESTE BLOQUE con la estrategia que elijan
# -------------------------------------------------------

# Variables a escalar (excluir targets y variables binarias)
# vars_escalar = ['edad', 'num_compras', 'ticket_promedio',
#                 'dias_ultima_compra', 'meses_cliente',
#                 'satisfaccion', 'pct_descuento_usado']

# Ejemplo con StandardScaler:
# scaler = StandardScaler()
# df_prep[vars_escalar] = scaler.fit_transform(df_prep[vars_escalar])

print('Estadísticos después del escalamiento:')
# print(df_prep[vars_escalar].describe().T)

## 3.5 Verificación Final del Dataset Procesado

> _(Verifiquen que el dataset está listo: sin missing values, variables codificadas, variables escaladas.)_

In [ ]:
print('=== RESUMEN DEL DATASET PROCESADO ===')
print(f'Dimensiones       : {df_prep.shape}')
print(f'Missing values    : {df_prep.isnull().sum().sum()}')
print(f'Tipos de datos    :')
print(df_prep.dtypes)
print()
df_prep.head()

---
# Conclusiones de la Entrega 1

> _(Escriban al menos 5 conclusiones relevantes que conecten los hallazgos técnicos con el contexto de negocio de RetailSmart. Ejemplo: qué descubrieron sobre el comportamiento de los clientes, qué variables parecen más relevantes, qué decisiones de preprocesamiento tomaron y por qué.)_

1. ...
2. ...
3. ...
4. ...
5. ...

---
*Machine Learning - MLY0100 | DUOC UC | Semestre 2025*